In [93]:
import json
import re

def parse_choice(item):
    """
    Returns (number, text) or (None, original) if not parseable.
    """
    item = str(item).strip()
    m = re.match(r"^(-?\d+)\s*(.*)$", item)
    if not m:
        return None, item
    return m.group(1), m.group(2).strip()


def is_redundant_scale(choices):
    """
    Detects cases like:
    ["1 A", "2 2", "3 3", "4 4", "5 B"]

    Condition:
    - all items start with number
    - all middle items are either:
        - "N N" or just "N"
    - at least one endpoint has descriptive text
    """
    parsed = [parse_choice(c) for c in choices]

    # must all be numeric-prefixed
    if any(num is None for num, _ in parsed):
        return False

    nums = [num for num, _ in parsed]

    # must be sequential-ish (weak check)
    if len(set(nums)) < 2:
        return False

    # detect redundancy pattern
    redundant = True
    for num, text in parsed[1:-1]:
        if text not in ("", num):
            redundant = False
            break

    return redundant


def transform_scale(choices):
    parsed = [parse_choice(c) for c in choices]

    result = []
    for i, (num, text) in enumerate(parsed):
        if text == "" or text == num:
            result.append(num)
        else:
            if i == 0 or i == len(parsed) - 1:
                result.append(f"{num} ({text})")
            else:
                result.append(num)
    return result


def process_jsonl(input_path, output_path):
    with open(input_path, "r") as infile, open(output_path, "w") as outfile:
        for line in infile:
            record = json.loads(line)

            for key, value in record.items():
                choices = value.get("choices")

                if isinstance(choices, list):
                    # step 1: remove negative-coded options
                    choices = [
                        c for c in choices
                        if not str(c).strip().startswith("-")
                    ]

                    # step 2: only transform if it's a redundant scale
                    if is_redundant_scale(choices):
                        value["choices"] = transform_scale(choices)
                    else:
                        value["choices"] = choices

            outfile.write(json.dumps(record) + "\n")

In [94]:
input_path = '../data/parsed_survey_data.jsonl'
output_path = '../data/parsed_survey_data_cleaned.jsonl'
process_jsonl(input_path, output_path)